# SGP Kits Statistics Report

This notebook analyzes the normalized kit statistics extracted from `command_storage.dat`.

The main focus is kit-level performance. Player IDs are kept as an explanatory dimension so we can detect cases where a kit's aggregate kill or damage output is heavily driven by one or a few players.

## Setup

The companion `sgp_data.py` module handles loading, validation, and derived datasets. `sgp_report.py` defines the SGP-specific figures, while `sgp_plot_components.py` contains their reusable interactive mechanics. This keeps the notebook focused on the analysis itself.


In [ ]:
from pathlib import Path
from plotly.offline import init_notebook_mode  

# Initialize Plotly offline mode for Jupyter Notebook  
init_notebook_mode(connected=True)

from sgp_data import load_report_data
from sgp_report import (
    ability_effectiveness_figure,
    ability_uses_figure,
    damage_causes_figure,
    damage_figure,
    elo_adjusted_kill_results_figure,
    kill_causes_figure,
    kill_concentration_figure,
    kill_concentration_scatter_figure,
    kills_vs_ability_uses_figure,
    matchup_figure,
    player_reach_figure,
    popularity_efficiency_figure,
    report_summary_figure,
    show_cause_profile_figure,
    show_player_contribution_figure,
    top_killer_exposure_figure,
    total_kills_figure,
)


report = load_report_data(Path("data"))


# Kit adoption

## Proportion of players who tried each kit

The denominator is every player appearing anywhere in the extracted kill, damage, ability-use, or kit-exposure data.

The default view uses observed playtime directly. Each wide, faded bar is the proportion who tried the kit; the narrower solid bar inside it is the proportion who also made at least one player kill. Ability reach remains available on hover.

The **Exposure shares** mode shows playtime and completed lives as two 100% stacked bars. Both use the same kit order, sorted by playtime share, so their segment sizes can be compared directly. **Life duration** shows average observed time per completed life.

These views distinguish broad player adoption from how heavily and how repeatedly each kit was played. Per-completed-life values use the aggregate counter as their denominator, so an unfinished current life is not counted as completed.

In [ ]:
fig = player_reach_figure(report)
fig.show()


# Kills

## Total kills by kit

Switch among total kills, the same totals stacked by player contribution, kills per active hour, kills per completed life, and kills per player-caused death. The **Kills / PvP death** mode compares attributed kills made with a kit against deaths attributed to a player while using that kit; its black line marks parity at 1.0, and non-player deaths remain available on hover. Rate hovers compare the server-wide result with the median eligible player's rate. In the kills-per-life mode, the thick black line marks 1.0 player kill per completed life and the dashed line marks the kit median. Player IDs appear on hover in the stacked view; click a segment to emphasize that player across every bar, then click it again to restore all colors.

In [ ]:
fig = total_kills_figure(report)
show_player_contribution_figure(fig)


## Kill causes as offensive identity and defensive vulnerability

The **Outgoing share** mode shows how each attacking kit delivers its attributed player kills. A kit dominated by one cause has a narrow offensive identity; a mixed profile indicates that several damage mechanisms regularly finish opponents. Only deaths with a real attacking kit and victim kit are included in this mode.

The two incoming modes reverse the perspective. **Incoming deaths / hour** compares each victim kit's exposure-normalized death rate and stacks the causes responsible; **Incoming share** removes the rate magnitude so cause susceptibility can be compared directly. These modes include deaths with no player killer, but exclude deaths where the victim had no kit. Cause labels use the names stored with the statistics. Click a segment to emphasize that cause across every kit and mode, then click it again to restore all causes.

In [ ]:
fig = kill_causes_figure(report)
show_cause_profile_figure(fig)


## Playtime share vs. kill efficiency

This separates popularity from exposure-normalized kill output. The vertical dashed line is the equal-share benchmark (one twelfth of kit playtime); the horizontal dashed line is the edition's overall observed kills per active hour. A point above the horizontal line produced kills faster than the server-wide rate, while its horizontal position shows whether players devoted more or less than an equal share of kit playtime to it.

This is still an FFA output measure, not a pure combat win rate: play style, map behavior, and the kinds of fights a kit takes can also affect kills per hour.

In [ ]:
fig = popularity_efficiency_figure(report)
fig.show()


## Player concentration of output and exposure

Each bar partitions a selected kit total into the top player, players 2–3, and everyone else. Switch among kills, attributed damage dealt, all damage received, playtime, and completed lives. The contributor count above each bar gives the context needed to interpret a concentrated result; the received-damage mode groups target players and includes non-player sources.

In [ ]:
fig = kill_concentration_figure(report)
fig.show()


## Total kills vs. player concentration

Each point is a kit. The dashed median lines divide the plot into four descriptive quadrants, separating kill volume from how strongly the result depends on the top player. Hover also shows playtime, completed lives, kills per hour, and kills per completed life.

In [ ]:
fig = kill_concentration_scatter_figure(report)
fig.show()


## Leading-player output relative to exposure

Use the buttons to select either the player with the most kills or the player with the most playtime for each kit. In both modes, the x-axis is the selected player's share of the kit's playtime and the y-axis is that same player's share of its kills. Above the diagonal, their kill share exceeds their exposure share; below it, playtime explains more of their apparent output dominance.

Hover identifies both selected players and whether they are the same person. This makes the concentration result easier to interpret without treating every top-player share as equally suspicious. Kits without valid shares for a mode are omitted from that mode.

In [ ]:
fig = top_killer_exposure_figure(report)
fig.show()


# Damage

Damage measures cumulative pressure in hearts, including damage that is later healed, so it complements rather than replaces the kill results.

## Damage output, intake, and player contribution

Switch among attributed player damage dealt, the same output stacked by source player, dealt per active hour, all damage received per active hour, and PvP damage exchange. Offensive modes require a real source kit and target kit. The received mode includes damage from every source when the target had a real kit; hover separates player and non-player damage. The exchange mode uses player-attributed damage in both directions, and its black line marks parity at 1.0. Player segments support the same click-to-focus interaction as the kill and ability plots.

In [ ]:
fig = damage_figure(report)
show_player_contribution_figure(fig)


## Damage causes as offensive identity and defensive pressure

The dealt modes show which causes make up each kit's attributed player damage, either as an exposure-normalized rate or a within-kit share. The received modes reverse the perspective and include all damage taken with a real target kit. Rate modes compare pressure magnitude; share modes isolate the mechanism mix. Exact values in hearts and player-versus-non-player context remain in hover information. Click a segment to emphasize that cause across every kit and mode, then click it again to restore all causes.

In [ ]:
fig = damage_causes_figure(report)
show_cause_profile_figure(fig)


# Matchups

These views describe observed kills and attributed damage between kits. They are not engagement win-rate estimates because the dataset does not contain fights that ended without a logged kill or time played in each matchup; damage also includes pressure that may later be healed.

## Directional kill and damage shares, plus raw matrices

The default kill-share view uses one cell per kit pair. A value above 50% means the row kit killed the column kit more often than the reverse. **Kills vs Elo** subtracts the share implied by the two players' current Kill Elo ratings from that observed share; positive cells favor the row kit after this player-skill context. This uses the extraction-time rating snapshot, not each player's rating immediately before a historical kill. The damage-share mode applies the directional comparison to cumulative attributed damage. Hover reveals exact shares and pair volume. Raw kill and damage matrices remain available behind buttons; damage and Elo-adjusted cells keep detailed values in hover rather than printing them into the heatmap.

In [ ]:
fig = matchup_figure(
    report.matchup_matrix,
    report.directional_share,
    report.pair_totals,
    report.matchup_kills_by_cause,
    elo_matchup_expected_share=report.elo_matchup_expected_share,
    elo_matchup_score_difference=report.elo_matchup_score_difference,
    elo_matchup_pair_totals=report.elo_matchup_pair_totals,
    elo_name=(
        str(report.elo_metadata["elo_name"].iloc[0])
        if not report.elo_metadata.empty
        else "Kill Elo"
    ),
    damage_matchup_matrix=report.damage_matchup_matrix,
    damage_directional_share=report.damage_directional_share,
    damage_pair_totals=report.damage_pair_totals,
    matchup_damage_by_cause=report.matchup_damage_by_cause,
)
fig.show()


# Abilities

Each kit has one logged ability. Activation counts show engagement, while successful uses and the kit-specific effect metric describe whether activations achieved their intended result. Because cooldowns differ by kit, raw uses per hour remain context rather than the main cross-kit intensity comparison.

## Ability uses by kit

Switch among total uses, player-stacked contributions, cooldown-normalized use, successful-use rate, and uses per completed life. **Cooldown-normalized use** divides the observed uses-per-hour rate by the maximum rate implied by the configured cooldown alone. For example, 25% means one activation for roughly every four cooldown lengths of playtime. It is not a combat-opportunity percentage: travel and downtime remain in the denominator, and cooldown resets can make values exceed 100%.

A successful use follows the kit-specific condition documented in hover; it does not mean the ability caused a kill. Poseidon's Splash currently has no success or effect metric and is therefore absent from success-based modes. Kit-specific effect totals and effect per successful use remain in hover. Player IDs appear in the stacked view; click a segment to emphasize that player across every bar, then click it again to restore all colors.

In [ ]:
fig = ability_uses_figure(report)
show_player_contribution_figure(fig)


## Ability engagement and effectiveness

The default mode separates how often an ability is activated from how often it meets its own success condition. The other modes compare effect magnitude only where the metadata provides a genuinely shared unit: affected players per successful cast, or health impact in hearts per successful use. Median guides and quadrants are recalculated for each eligible subset.

The hearts mode combines offensive damage for Tank and Cancer with resisted damage for Enderman. It is a common magnitude scale, not a claim that dealing and resisting damage have identical balance value. Pigeon's lock time, Archer's displacement, and Alchemist's destroyed decoys have unique units, so their values stay in hover rather than being placed on a misleading shared axis.

In [ ]:
fig = ability_effectiveness_figure(report)
fig.show()


# Combined analysis

## Kills vs. ability uses

Each point is a kit. Switch among aggregate totals, cooldown-normalized ability use against kills per hour, successful-use rate against kills per hour, and per-completed-life rates; median lines and quadrant labels are recalculated for every mode. The usage modes answer a different question from the effectiveness plot above: whether observed ability engagement is associated with kill output. None of these comparisons implies that ability use or success causes kills.

In [ ]:
fig = kills_vs_ability_uses_figure(report.combined_totals)
fig.show()


## Kill results relative to current player Elo

Each credited cross-kit PvP kill is treated as a binary result: one score for the killer's kit and zero for the victim's kit. The plot compares each kit's observed score share with the score implied by the two players' current Kill Elo ratings, then sorts kits by the difference. Same-kit kills are excluded because they add one win and one loss to the same kit and carry no kit-balance signal.

This is a player-skill context, not a reconstructed historical Elo expectation: the extraction contains current ratings rather than each player's rating immediately before every kill. Result volume and both underlying shares remain in hover.

In [ ]:
fig = elo_adjusted_kill_results_figure(report)
fig.show()


# Summary

The final plot aligns four complementary views of every kit while keeping each metric on a meaningful scale. Kits are sorted by total kills. The kill hover includes deaths per hour, K/D, and the non-player death share. The ability panel keeps cooldown-normalized activation intensity; successful-use rate and kit-specific effect per success are added to hover. The reach panel distinguishes actual play, ability use, and making a kill. Vertical guides show edition medians, with the reach guides corresponding to actual play and kill reach.

In [ ]:
fig = report_summary_figure(report)
fig.show()
